### 📑 `description=` vs `docstring` —— 两个看起来一样的"描述",读者却不同

这是 `@tool` 装饰器**最容易混淆**的一点。下面这段代码里**两段文字**看似都在描述同一个函数,其实**目标读者不同**:

```python
@tool(description=LS_DESCRIPTION)                         # ← 🤖 给 LLM 看的
def ls(state: Annotated[DeepAgentState, InjectedState]) -> list[str]:
    """List all files in the virtual filesystem."""      # ← 👨‍💻 给程序员看的
    return list(state.get("files", {}).keys())
```

#### 🎯 两个字段的分工

| 字段 | 目标读者 | 干什么用 |
|---|---|---|
| `description=...` | 🤖 **LLM** | 工具描述 → 让 LLM 知道**何时 / 如何**调用 |
| `"""docstring"""` | 👨‍💻 **程序员 / IDE** | 代码可读性、`help()`、IDE 悬浮提示、Sphinx 文档 |

#### 🔁 优先级规则(关键!)

| 情况 | LLM 看到的描述 | 程序员看到的 docstring |
|---|---|---|
| ✅ 只有 `description=` | `description=` 的值 | docstring(独立) |
| ✅ 只有 docstring | **docstring**(自动当 LLM 描述用)| 同 docstring |
| ⚠️ **两个都有**(本节的情况) | **`description=` 的值** | docstring |
| ❌ 两个都没有 | 报错 —— 工具必须有描述 | — |

> 🔑 **核心规则**:`description=` 一旦提供,就**完全接管** LLM 端的描述。docstring 不会"补充"它,而是被 LLM **彻底忽略**(但程序员仍能看到)。

#### 💡 那为什么本节要两个都写?

```python
@tool(description=LS_DESCRIPTION)        # 几十行长 prompt(教学指令)
def ls(state):
    """List all files in the virtual filesystem."""   # 一句话(语义摘要)
```

- 📜 `LS_DESCRIPTION` 是从 `prompts.py` 导入的 **几十行长 prompt**(教 LLM 何时 `ls`、和其他工具怎么配合)—— 太长的 prompt 不适合塞进 docstring
- 📝 docstring 是 **一句话语义摘要**,打开 IDE 时一目了然

**两边各取所需**:LLM 拿教学指令,程序员拿精炼摘要,互不打架。

#### 🔧 一个例外:`parse_docstring=True`

注意 **下一段代码** 里 `read_file` / `write_file` 多了个参数:

```python
@tool(description=READ_FILE_DESCRIPTION, parse_docstring=True)
def read_file(file_path, state, offset=0, limit=2000):
    """Read file content...

    Args:
        file_path: Path to the file to read
        offset:    Line number to start reading from (default: 0)
        ...
    """
```

`parse_docstring=True` 让 LangChain **从 docstring 的 `Args:` 段抽取每个参数的说明**,塞进 LLM 看到的工具 schema。所以这种情况下:

| 谁 | 看到什么 |
|---|---|
| 🤖 **LLM** | `description=` 的整体描述 **+** `Args:` 段抽出的参数说明 |
| 👨‍💻 **程序员** | 完整 docstring |

> ⚠️ 这是 docstring **唯一**能被 LLM 看见的情况,而且**仅限 `Args:` 段**(函数的一句话总结仍然被忽略)。